# X-Forwarded-For (XFF) Header Monitoring — Customer Guidance

## Background & Problem Statement

In a typical Azure multi-tier architecture — **Proxy (VIP) → Application Gateway → App Service** — the **real client IP is lost** at each tier because each service only logs the **TCP peer IP** (the immediate upstream connection), not the original client.

### Customer Questions Addressed

| # | Question |
|---|---------|
| **Q1** | Since App Gateway `AGWAccessLogs.ClientIp` does not capture the XFF header, what is an alternative approach to store X-Forwarded-For at the App Gateway level? |
| **Q2** | Since App Service `AppServiceHTTPLogs.CIp` does not capture the XFF header, what is an alternative approach to store X-Forwarded-For at the App Service level? |
| **Q3** | How many days/months can we store these logs in Diagnostic Logs, and what is the cost? |

> **This notebook provides visual evidence from a live test environment, architecture diagrams, code samples, and KQL queries to answer all three questions.**

In [ ]:
# ── Setup: Import libraries & define KQL query helper ─────────────────────────
import subprocess, json, os
import pandas as pd
from IPython.display import display, HTML, SVG, Markdown
from datetime import datetime

# Log Analytics workspace ID (from test environment)
WORKSPACE_ID = "e5f08bd4-cd12-4ab3-ae87-cfb8bc4738cf"

def run_kql(query: str, workspace_id: str = WORKSPACE_ID) -> pd.DataFrame:
    """Run a KQL query against Log Analytics and return as DataFrame."""
    cmd = [
        "az", "monitor", "log-analytics", "query",
        "-w", workspace_id,
        "--analytics-query", query,
        "-o", "json"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    if result.returncode != 0:
        print(f"⚠️ Query error: {result.stderr.strip()}")
        return pd.DataFrame()
    data = json.loads(result.stdout)
    df = pd.DataFrame(data)
    df = df.drop(columns=["TableName"], errors="ignore")
    return df

print(f"✅ Setup complete — {datetime.now().strftime('%Y-%m-%d %H:%M')} | Workspace: {WORKSPACE_ID}")

## Architecture — Request Flow & XFF Header at Each Hop

The diagram below shows how requests flow through the multi-tier architecture and how the `X-Forwarded-For` header changes at each hop. **The key insight: native log fields only capture the TCP peer, not the real client IP.**

In [ ]:
# ── Architecture Diagram: XFF Header Flow (Draw.io style SVG) ─────────────────
display(HTML("""
<div style="background:#fff;border:2px solid #e0e0e0;border-radius:12px;padding:24px;max-width:1050px;font-family:Segoe UI,sans-serif;">

<!-- Title -->
<div style="text-align:center;font-size:18px;font-weight:700;color:#1a1a1a;margin-bottom:18px;">
  Proxy (VIP) → Application Gateway → App Service — XFF Header Flow
</div>

<!-- Infrastructure Flow -->
<div style="display:flex;align-items:center;justify-content:center;gap:10px;margin-bottom:8px;">

  <!-- Client -->
  <div style="background:#424242;color:#fff;border-radius:10px;padding:14px 18px;text-align:center;min-width:100px;box-shadow:0 3px 8px rgba(0,0,0,.15);">
    <div style="font-weight:700;font-size:13px;">👤 Client</div>
    <div style="font-size:11px;color:#ccc;margin-top:4px;">203.0.113.50</div>
  </div>

  <div style="font-size:24px;color:#666;">→</div>

  <!-- Proxy -->
  <div style="background:#F25022;color:#fff;border-radius:10px;padding:14px 18px;text-align:center;min-width:120px;box-shadow:0 3px 8px rgba(242,80,34,.3);">
    <div style="font-weight:700;font-size:13px;">🔄 Proxy (VIP)</div>
    <div style="font-size:11px;color:#ffe0b2;margin-top:2px;">Nginx on ACI</div>
    <div style="font-size:10px;color:#ffcc80;margin-top:2px;">IP: 4.174.181.251</div>
  </div>

  <div style="font-size:24px;color:#666;">→</div>

  <!-- App Gateway -->
  <div style="background:#0078D4;color:#fff;border-radius:10px;padding:14px 18px;text-align:center;min-width:140px;box-shadow:0 3px 8px rgba(0,120,212,.3);position:relative;">
    <div style="font-weight:700;font-size:13px;">🛡️ App Gateway</div>
    <div style="font-size:11px;color:#bbdefb;margin-top:2px;">Basic SKU</div>
    <div style="font-size:10px;color:#90caf9;margin-top:2px;">IP: 4.205.85.213</div>
    <div style="position:absolute;top:-32px;left:50%;transform:translateX(-50%);background:#E3F2FD;color:#1565C0;
         font-size:9px;padding:3px 8px;border-radius:4px;border:1px solid #90CAF9;white-space:nowrap;">
      Rewrite: {var_add_x_forwarded_for_proxy}
    </div>
  </div>

  <div style="font-size:24px;color:#666;">→</div>

  <!-- App Service -->
  <div style="background:#7FBA00;color:#fff;border-radius:10px;padding:14px 18px;text-align:center;min-width:120px;box-shadow:0 3px 8px rgba(127,186,0,.3);position:relative;">
    <div style="font-weight:700;font-size:13px;">🌐 App Service</div>
    <div style="font-size:11px;color:#dcedc8;margin-top:2px;">Linux / .NET 8</div>
    <div style="font-size:10px;color:#c5e1a5;margin-top:2px;">F1 Free Tier</div>
    <div style="position:absolute;top:-32px;left:50%;transform:translateX(-50%);background:#E8F5E9;color:#2E7D32;
         font-size:9px;padding:3px 8px;border-radius:4px;border:1px solid #A5D6A7;white-space:nowrap;">
      ForwardedHeaders + XffTelemetry
    </div>
  </div>
</div>

<!-- XFF Header Values -->
<div style="margin-top:20px;font-size:12px;font-weight:600;text-align:center;color:#555;margin-bottom:8px;">
  X-Forwarded-For Header at Each Hop
</div>
<div style="display:flex;justify-content:center;gap:8px;">
  <div style="background:#FFF3E0;border:1px solid #FF9800;border-radius:6px;padding:8px 12px;text-align:center;flex:1;max-width:150px;">
    <div style="font-size:10px;font-weight:600;color:#E65100;">Before Proxy</div>
    <div style="font-size:11px;color:#999;font-style:italic;">(none)</div>
  </div>
  <div style="color:#FF9800;font-size:18px;align-self:center;">→</div>
  <div style="background:#FFF3E0;border:1px solid #FF9800;border-radius:6px;padding:8px 12px;text-align:center;flex:1;max-width:180px;">
    <div style="font-size:10px;font-weight:600;color:#E65100;">After Proxy</div>
    <div style="font-size:11px;color:#E65100;font-weight:600;">203.0.113.50</div>
  </div>
  <div style="color:#FF9800;font-size:18px;align-self:center;">→</div>
  <div style="background:#FFF3E0;border:1px solid #FF9800;border-radius:6px;padding:8px 12px;text-align:center;flex:1;max-width:250px;">
    <div style="font-size:10px;font-weight:600;color:#E65100;">After App Gateway</div>
    <div style="font-size:11px;color:#E65100;font-weight:600;">203.0.113.50, 4.172.218.37</div>
  </div>
  <div style="color:#FF9800;font-size:18px;align-self:center;">→</div>
  <div style="background:#FFF3E0;border:1px solid #FF9800;border-radius:6px;padding:8px 12px;text-align:center;flex:1;max-width:250px;">
    <div style="font-size:10px;font-weight:600;color:#E65100;">At App Service</div>
    <div style="font-size:11px;color:#E65100;font-weight:600;">203.0.113.50, 4.172.218.37</div>
    <div style="font-size:9px;color:#999;">(read by middleware)</div>
  </div>
</div>

</div>
"""))

---
## Q1: How to Store XFF at the Application Gateway Level?

### The Limitation
`AGWAccessLogs.ClientIp` (resource-specific mode) or `AzureDiagnostics.clientIP_s` (legacy mode) **always records the TCP peer IP** — the IP of the device directly connected to App Gateway. When a proxy sits in front, this is the proxy IP, not the real client.

**There is no native column in AGWAccessLogs that captures the full XFF header.**

### The Solution
1. **Deploy a Rewrite Rule** with `{var_add_x_forwarded_for_proxy}` — normalizes XFF and strips the non-standard `:port` suffix
2. **Capture XFF in Application Insights** on the backend using a telemetry initializer

### Evidence — What App Gateway Logs Actually Show

In [ ]:
# ── KQL: What IP does App Gateway see? ────────────────────────────────────────
query = """
AzureDiagnostics
| where Category == 'ApplicationGatewayAccessLog'
| where TimeGenerated > ago(48h)
| summarize Requests = count() by clientIP_s
| order by Requests desc
"""
df_agw = run_kql(query)
if not df_agw.empty:
    df_agw.columns = ["Client IP (as seen by App Gateway)", "Request Count"]
    display(Markdown("### App Gateway Access Logs — Client IP Distribution"))
    display(Markdown("> ⚠️ **Notice:** These are all **proxy/scanner IPs** — the real end-user IP is nowhere in this table."))
    display(df_agw.style
        .set_properties(**{"text-align": "left"})
        .bar(subset=["Request Count"], color="#FFCDD2")
        .set_caption("AzureDiagnostics | clientIP_s — Shows TCP peer, NOT real client")
    )

### App Gateway — Rewrite Rule Configuration

**Portal steps:**
1. Navigate to **Application Gateway → Rewrites**
2. Click **+ Rewrite set** → Name: `xff-normalization-ruleset`
3. Associate with your routing rule(s)
4. Add rule: Header `X-Forwarded-For` = `{var_add_x_forwarded_for_proxy}`

**Bicep (Infrastructure as Code):**
```bicep
resource rewriteRuleSet 'Microsoft.Network/applicationGateways/rewriteRuleSets@2023-11-01' = {
  name: 'xff-normalization-ruleset'
  parent: appGateway
  properties: {
    rewriteRules: [{
      name: 'Normalize-XFF'
      ruleSequence: 100
      conditions: []
      actionSet: {
        requestHeaderConfigurations: [{
          headerName: 'X-Forwarded-For'
          headerValue: '{var_add_x_forwarded_for_proxy}'
        }]
        responseHeaderConfigurations: []
      }
    }]
  }
}
```

| Server Variable | Description |
|----------------|-------------|
| `{var_add_x_forwarded_for_proxy}` | Full XFF chain with port stripped from last entry |
| `{var_client_ip}` | Direct client IP (TCP peer, no port) |

> ⚠️ **Important:** The rewrite rule set must be **associated with routing rules** to take effect.

---
## Q2: How to Store XFF at the App Service Level?

### The Limitation
`AppServiceHTTPLogs.CIp` **always shows the TCP peer IP** — in this architecture, that is the Application Gateway's IP. There is no native XFF column in `AppServiceHTTPLogs`.

### The Solution
1. **ForwardedHeaders Middleware** — resolves `RemoteIpAddress` from the XFF header
2. **XffTelemetryInitializer** — captures XFF into Application Insights `customDimensions`

### Evidence — What App Service Logs Actually Show

In [ ]:
# ── KQL: What IP does App Service see? ────────────────────────────────────────
query = """
AppServiceHTTPLogs
| where TimeGenerated > ago(48h)
| summarize Requests = count() by CIp
| order by Requests desc
| take 10
"""
df_appsvc = run_kql(query)
if not df_appsvc.empty:
    df_appsvc.columns = ["Client IP (as seen by App Service)", "Request Count"]
    total = df_appsvc["Request Count"].astype(int).sum()
    appgw_reqs = df_appsvc[df_appsvc["Client IP (as seen by App Service)"] == "4.205.85.213"]["Request Count"].astype(int).sum()
    pct = round(100 * appgw_reqs / total, 1) if total > 0 else 0

    display(Markdown("### App Service HTTP Logs — Client IP Distribution"))
    display(Markdown(f"> ⚠️ **{pct}% of requests** show `4.205.85.213` (App Gateway IP) as the client — **the real end-user IP is lost!**"))
    display(df_appsvc.style
        .set_properties(**{"text-align": "left"})
        .bar(subset=["Request Count"], color="#FFCDD2")
        .set_caption("AppServiceHTTPLogs | CIp — Shows TCP peer (App GW), NOT real client")
    )

In [ ]:
# ── Monitoring Data Flow Diagram ───────────────────────────────────────────────
display(HTML("""
<div style="background:#fff;border:2px solid #e0e0e0;border-radius:12px;padding:24px;max-width:1050px;font-family:Segoe UI,sans-serif;">

<!-- Title -->
<div style="text-align:center;font-size:17px;font-weight:700;color:#1a1a1a;margin-bottom:16px;">
  Where Monitoring Captures Data — And How to Get Real Client IP
</div>

<!-- Three log tables -->
<div style="display:flex;gap:14px;justify-content:center;">

  <!-- Table 1: AzureDiagnostics -->
  <div style="flex:1;max-width:280px;border:2px solid #D32F2F;border-radius:10px;padding:14px;background:#fff;">
    <div style="font-weight:700;font-size:12px;color:#333;margin-bottom:6px;">📋 AzureDiagnostics</div>
    <div style="font-size:10px;color:#888;margin-bottom:8px;">App Gateway Access Logs</div>
    <div style="background:#FFEBEE;border-radius:4px;padding:6px 8px;margin-bottom:6px;">
      <span style="font-size:11px;font-weight:600;">clientIP_s = </span>
      <span style="font-size:11px;color:#D32F2F;font-weight:700;">Proxy VIP IP ⚠️</span>
    </div>
    <div style="font-size:9px;color:#999;">TCP peer only — shows who connected<br>to App Gateway, NOT the real user</div>
    <div style="margin-top:8px;font-size:10px;color:#D32F2F;font-weight:600;">❌ Real client IP NOT available</div>
  </div>

  <!-- Table 2: AppServiceHTTPLogs -->
  <div style="flex:1;max-width:280px;border:2px solid #D32F2F;border-radius:10px;padding:14px;background:#fff;">
    <div style="font-weight:700;font-size:12px;color:#333;margin-bottom:6px;">📋 AppServiceHTTPLogs</div>
    <div style="font-size:10px;color:#888;margin-bottom:8px;">App Service Platform Logs</div>
    <div style="background:#FFEBEE;border-radius:4px;padding:6px 8px;margin-bottom:6px;">
      <span style="font-size:11px;font-weight:600;">CIp = </span>
      <span style="font-size:11px;color:#D32F2F;font-weight:700;">App Gateway IP ⚠️</span>
    </div>
    <div style="font-size:9px;color:#999;">TCP peer only — shows who connected<br>to App Service, NOT the real user</div>
    <div style="margin-top:8px;font-size:10px;color:#D32F2F;font-weight:600;">❌ Real client IP NOT available</div>
  </div>

  <!-- Table 3: Application Insights ✅ -->
  <div style="flex:1;max-width:310px;border:3px solid #2E7D32;border-radius:10px;padding:14px;background:#fff;box-shadow:0 2px 8px rgba(46,125,50,.15);">
    <div style="font-weight:700;font-size:12px;color:#333;margin-bottom:6px;">🔍 Application Insights</div>
    <div style="font-size:10px;color:#888;margin-bottom:8px;">requests table → customDimensions</div>
    <div style="background:#E8F5E9;border-radius:4px;padding:6px 8px;margin-bottom:4px;">
      <span style="font-size:10px;font-weight:600;">customDimensions["X-Forwarded-For"]</span>
    </div>
    <div style="background:#E8F5E9;border-radius:4px;padding:6px 8px;margin-bottom:6px;">
      <span style="font-size:11px;color:#2E7D32;font-weight:700;">= 203.0.113.50 (Real Client!) ✅</span>
    </div>
    <div style="font-size:9px;color:#999;">Captured by ForwardedHeaders middleware<br>+ XffTelemetryInitializer</div>
    <div style="margin-top:8px;font-size:10px;color:#2E7D32;font-weight:600;">✅ Real client IP AVAILABLE</div>
  </div>
</div>

<!-- Required Configuration Steps -->
<div style="margin-top:20px;padding-top:16px;border-top:2px dashed #9C27B0;">
  <div style="text-align:center;font-size:14px;font-weight:700;color:#6A1B9A;margin-bottom:12px;">
    Required Configuration for Real Client IP
  </div>
  <div style="display:flex;gap:10px;justify-content:center;">
    <div style="flex:1;max-width:200px;background:#FFF3E0;border:1px solid #F25022;border-radius:8px;padding:10px;text-align:center;">
      <div style="font-weight:700;font-size:12px;color:#E65100;">① Proxy</div>
      <div style="font-size:10px;color:#666;margin-top:4px;">Set X-Forwarded-For<br>on outbound requests</div>
      <div style="font-size:9px;color:#999;margin-top:4px;font-style:italic;">$proxy_add_x_forwarded_for</div>
    </div>
    <div style="align-self:center;color:#666;font-size:20px;">→</div>
    <div style="flex:1;max-width:200px;background:#E3F2FD;border:1px solid #0078D4;border-radius:8px;padding:10px;text-align:center;">
      <div style="font-weight:700;font-size:12px;color:#0D47A1;">② App Gateway</div>
      <div style="font-size:10px;color:#666;margin-top:4px;">Deploy Rewrite Rule</div>
      <div style="font-size:9px;color:#1565C0;margin-top:4px;font-style:italic;">{var_add_x_forwarded_for_proxy}</div>
    </div>
    <div style="align-self:center;color:#666;font-size:20px;">→</div>
    <div style="flex:1;max-width:200px;background:#E8F5E9;border:1px solid #7FBA00;border-radius:8px;padding:10px;text-align:center;">
      <div style="font-weight:700;font-size:12px;color:#33691E;">③ App Service</div>
      <div style="font-size:10px;color:#666;margin-top:4px;">ForwardedHeaders Middleware<br>(must be FIRST!)</div>
    </div>
    <div style="align-self:center;color:#666;font-size:20px;">→</div>
    <div style="flex:1;max-width:200px;background:#F3E5F5;border:1px solid #9C27B0;border-radius:8px;padding:10px;text-align:center;">
      <div style="font-weight:700;font-size:12px;color:#6A1B9A;">④ Telemetry Init</div>
      <div style="font-size:10px;color:#666;margin-top:4px;">XffTelemetryInitializer<br>→ customDimensions</div>
    </div>
  </div>
</div>

</div>
"""))

### App Service — ForwardedHeaders Middleware + Telemetry Initializer

**.NET (ASP.NET Core):**
```csharp
// Step 1: Configure ForwardedHeaders (Program.cs)
builder.Services.Configure<ForwardedHeadersOptions>(options =>
{
    options.ForwardedHeaders = ForwardedHeaders.XForwardedFor | ForwardedHeaders.XForwardedProto;
    options.KnownNetworks.Add(new IPNetwork(IPAddress.Parse("10.0.0.0"), 8));
    options.ForwardLimit = null;  // Multi-proxy chains
});

app.UseForwardedHeaders();  // ← MUST be FIRST middleware

// Step 2: Register XFF Telemetry Initializer
builder.Services.AddApplicationInsightsTelemetry();
builder.Services.AddSingleton<ITelemetryInitializer, XffTelemetryInitializer>();
```

**Python (Flask):**
```python
from werkzeug.middleware.proxy_fix import ProxyFix
app.wsgi_app = ProxyFix(app.wsgi_app, x_for=2, x_proto=1, x_host=1)
```

**Java (Spring Boot):**
```properties
server.forward-headers-strategy=FRAMEWORK
```

### KQL to Query Real Client IP (after middleware deployed)
```kql
requests
| where timestamp > ago(24h)
| extend xff = tostring(customDimensions["X-Forwarded-For"])
| project timestamp,
    RealClientIp = split(xff, ",")[0],
    FullXffChain = xff,
    AppServiceCIp = client_IP,
    url, resultCode
| order by timestamp desc
```

---
## Q3: Log Retention — Duration & Cost

Azure Log Analytics supports flexible retention with tiered pricing:

In [ ]:
# ── Retention & Cost Visualization ─────────────────────────────────────────────
display(HTML("""
<div style="background:#fff;border:2px solid #e0e0e0;border-radius:12px;padding:24px;max-width:1050px;font-family:Segoe UI,sans-serif;">

<div style="text-align:center;font-size:17px;font-weight:700;color:#1a1a1a;margin-bottom:16px;">
  📦 Log Retention Options & Cost
</div>

<!-- Retention Tiers -->
<div style="display:flex;gap:12px;justify-content:center;margin-bottom:20px;">

  <div style="flex:1;max-width:280px;border-radius:10px;padding:16px;background:linear-gradient(135deg,#E3F2FD,#BBDEFB);border:1px solid #64B5F6;">
    <div style="font-weight:700;font-size:14px;color:#0D47A1;">🔥 Interactive (Hot)</div>
    <div style="font-size:24px;font-weight:700;color:#1565C0;margin:8px 0;">30 — 730 days</div>
    <div style="font-size:11px;color:#333;">Fully queryable with KQL</div>
    <div style="margin-top:10px;background:#fff;border-radius:6px;padding:8px;font-size:11px;">
      <b>Cost:</b> Free for first 31 days<br>
      After: <b>$0.10/GB/month</b>
    </div>
  </div>

  <div style="flex:1;max-width:280px;border-radius:10px;padding:16px;background:linear-gradient(135deg,#FFF3E0,#FFE0B2);border:1px solid #FFB74D;">
    <div style="font-weight:700;font-size:14px;color:#E65100;">❄️ Archive (Cold)</div>
    <div style="font-size:24px;font-weight:700;color:#FF6D00;margin:8px 0;">Up to 12 years</div>
    <div style="font-size:11px;color:#333;">Search-only access</div>
    <div style="margin-top:10px;background:#fff;border-radius:6px;padding:8px;font-size:11px;">
      <b>Cost:</b> <b>$0.02/GB/month</b><br>
      + $0.01/GB search query
    </div>
  </div>

  <div style="flex:1;max-width:280px;border-radius:10px;padding:16px;background:linear-gradient(135deg,#E8F5E9,#C8E6C9);border:1px solid #81C784;">
    <div style="font-weight:700;font-size:14px;color:#2E7D32;">💾 Export to Storage</div>
    <div style="font-size:24px;font-weight:700;color:#388E3C;margin:8px 0;">Unlimited</div>
    <div style="font-size:11px;color:#333;">Cheapest long-term option</div>
    <div style="margin-top:10px;background:#fff;border-radius:6px;padding:8px;font-size:11px;">
      <b>Cost:</b> <b>$0.018/GB/month</b> (Hot)<br>
      $0.01/GB/month (Cool)
    </div>
  </div>
</div>

<!-- Ingestion Cost -->
<div style="background:#F5F5F5;border-radius:8px;padding:14px;margin-bottom:16px;">
  <div style="font-weight:700;font-size:13px;color:#333;margin-bottom:8px;">💰 Data Ingestion Cost</div>
  <div style="display:flex;gap:20px;">
    <div style="font-size:12px;"><b>Analytics plan:</b> $2.76/GB &nbsp;|&nbsp; <b>Basic Logs:</b> $0.65/GB &nbsp;|&nbsp; <b>Free allowance:</b> 5 GB/day per billing account</div>
  </div>
</div>

<!-- Cost Examples -->
<table style="width:100%;border-collapse:collapse;font-size:12px;text-align:left;">
  <tr style="background:#ECEFF1;">
    <th style="padding:8px;border:1px solid #CFD8DC;">Daily Ingestion</th>
    <th style="padding:8px;border:1px solid #CFD8DC;">Retention</th>
    <th style="padding:8px;border:1px solid #CFD8DC;">Monthly Cost (approx.)</th>
  </tr>
  <tr><td style="padding:6px;border:1px solid #eee;">1 GB/day</td><td style="padding:6px;border:1px solid #eee;">30 days</td><td style="padding:6px;border:1px solid #eee;">~$83/month</td></tr>
  <tr style="background:#F9FBE7;"><td style="padding:6px;border:1px solid #eee;">1 GB/day</td><td style="padding:6px;border:1px solid #eee;">90 days</td><td style="padding:6px;border:1px solid #eee;">~$89/month</td></tr>
  <tr><td style="padding:6px;border:1px solid #eee;">1 GB/day</td><td style="padding:6px;border:1px solid #eee;">365 days</td><td style="padding:6px;border:1px solid #eee;">~$103/month</td></tr>
  <tr style="background:#E8F5E9;"><td style="padding:6px;border:1px solid #eee;"><b>≤5 GB/day</b></td><td style="padding:6px;border:1px solid #eee;">30 days</td><td style="padding:6px;border:1px solid #eee;"><b>FREE</b> (5 GB allowance)</td></tr>
  <tr><td style="padding:6px;border:1px solid #eee;">5 GB/day</td><td style="padding:6px;border:1px solid #eee;">90 days</td><td style="padding:6px;border:1px solid #eee;">~$30/month (retention only)</td></tr>
  <tr style="background:#FFEBEE;"><td style="padding:6px;border:1px solid #eee;">10 GB/day</td><td style="padding:6px;border:1px solid #eee;">90 days</td><td style="padding:6px;border:1px solid #eee;">~$198/month</td></tr>
</table>

</div>
"""))

In [ ]:
# ── KQL: Current data ingestion volume ────────────────────────────────────────
query = """
Usage
| where TimeGenerated > ago(48h)
| summarize DataIngested_MB = round(sum(Quantity), 2) by DataType
| order by DataIngested_MB desc
"""
df_usage = run_kql(query)
if not df_usage.empty:
    df_usage.columns = ["Data Type", "Ingested (MB)"]
    display(Markdown("### Current Data Ingestion (Last 48 Hours)"))
    display(df_usage.style
        .bar(subset=["Ingested (MB)"], color="#C8E6C9")
        .set_caption("Usage | DataType — Actual ingestion volume in this test environment")
    )

### Configure Retention (CLI)
```bash
# Set 90-day interactive retention + 1 year archive
az monitor log-analytics workspace table update \
  --resource-group <rg> --workspace-name <workspace> \
  --name AppServiceHTTPLogs --retention-time 90 --total-retention-time 365

az monitor log-analytics workspace table update \
  --resource-group <rg> --workspace-name <workspace> \
  --name AzureDiagnostics --retention-time 90 --total-retention-time 365
```

| Scenario | Recommended Retention | Extra Cost |
|----------|----------------------|------------|
| **Standard operations** | 90 days interactive | ~$0.10/GB/month for days 32–90 |
| **Security & compliance** | 90 days interactive + 1 yr archive | + $0.02/GB/month archive |
| **Cost-sensitive** | 30 days default + export to Storage | ~$0.018/GB/month storage |

In [ ]:
# ── In-Memory Summary: Test Environment Metadata ──────────────────────────────
env_data = {
    "Component": ["App Gateway", "ACI Proxy (Nginx)", "App Service", "Log Analytics", "App Insights"],
    "Resource": [
        "appgw-xff-test-cg4l2wl4myrww", "aci-proxy-cg4l2wl4myrww",
        "app-xff-test-cg4l2wl4myrww", "log-xff-test-cg4l2wl4myrww", "ai-xff-test-cg4l2wl4myrww"
    ],
    "SKU / Config": ["Basic (Gen1)", "1 vCPU, 1.5 GB", "F1 Free / Linux", "PerGB2018, 30d", "Workspace-based"],
    "Public IP": ["4.205.85.213", "4.174.181.251", "*.azurewebsites.net", "—", "—"],
    "Daily Cost": ["~$0.77", "~$1.08", "Free", "~$0.03", "Free (5GB)"],
    "Status": ["✅ Running", "✅ Running", "✅ Running", "✅ Active", "✅ Active"]
}
df_env = pd.DataFrame(env_data)

display(Markdown("---\n## Test Environment Summary"))
display(Markdown(f"> **Region:** canadacentral &nbsp;|&nbsp; **Resource Group:** rg-xff-test-eastus &nbsp;|&nbsp; **Total Cost:** ~$2.00/day (~$60/month)"))
display(df_env.style
    .set_properties(**{"text-align": "left", "font-size": "12px"})
    .set_properties(subset=["Status"], **{"font-weight": "bold"})
    .set_caption("Live test environment — Proxy (VIP) → App Gateway → App Service")
)

---
## Implementation Checklist

### At the Proxy Layer
- [ ] Configure the proxy to add `X-Forwarded-For: <real-client-ip>` on outbound requests

### At the App Gateway Layer
- [ ] Deploy XFF normalization rewrite rule (`{var_add_x_forwarded_for_proxy}`)
- [ ] Associate rewrite rule set with all routing rules
- [ ] Enable diagnostic settings → Log Analytics workspace

### At the App Service Layer
- [ ] Deploy ForwardedHeaders middleware (**must be first middleware**)
- [ ] Configure `KnownNetworks` to trust App Gateway / proxy subnets
- [ ] Set `ForwardLimit = null` for multi-proxy chains
- [ ] Register `XffTelemetryInitializer` for Application Insights
- [ ] Enable diagnostic settings → Log Analytics workspace

### Monitoring & Retention
- [ ] Set per-table retention based on compliance needs
- [ ] Configure archive tier for long-term retention
- [ ] Set up data export to Storage Account if >1 year needed
- [ ] Deploy Azure Monitor Workbook for ongoing visibility

---
## Related Resources

| Resource | Location |
|----------|----------|
| XFF Overview | `docs/xff-overview.md` |
| App Gateway XFF Guide | `docs/xff-azure-application-gateway.md` |
| App Service XFF Guide | `docs/xff-azure-app-service.md` |
| Multi-Tier Patterns | `docs/xff-multi-tier-patterns.md` |
| .NET Sample Code | `samples/dotnet/Program.cs` |
| Python Middleware | `samples/python/xff_middleware.py` |
| KQL Validation Queries | `queries/xff-proxy-appgw-validation.kql` |
| Architecture Diagram (Draw.io) | `docs/diagrams/xff-architecture-flow.drawio` |
| Monitoring Flow Diagram (Draw.io) | `docs/diagrams/xff-monitoring-data-flow.drawio` |
| Azure Monitor Workbook | `workbook/xff-proxy-appgw-demo-workbook.json` |
| [Azure Monitor Pricing](https://azure.microsoft.com/pricing/details/monitor/) | Microsoft Docs |
| [ForwardedHeaders Docs](https://learn.microsoft.com/aspnet/core/host-and-deploy/proxy-load-balancer) | Microsoft Learn |